# Cloud heights at every site

Where are the clouds that enter the susceptibility fits, and does that change the answer? For each site with cloud heights this notebook:

1. tabulates cloud base, cloud top and thickness (median and IQR), and how many hours the standard height screen (base ≤ 2 km, top ≤ 3 km) removes;
2. plots the base and top distributions side by side, their histograms, and the seasonal cycle of cloud base;
3. shows how coupled the cloud is to the surface (cloud base minus surface LCL) at the ARM sites;
4. refits S(100 nm) in cloud-base terciles at each site, to see whether low and high clouds respond differently;
5. for the aircraft campaigns, shows the cloud base and how far above base the cloud passes were.

**Height sources differ.** ARM sites: cloud base/top from the NDROP files (ARSCL, metres above ground). Cloudnet sites: lowest/highest *reliable liquid* pixel above ground, so the top is the top of the retrievable liquid, not necessarily the cloud top. Aircraft: metres above sea level. The in-situ stations (Puijo, Pallas, Zeppelin) have no cloud heights; the cloud is at the station.

Run it from the folder that holds `results/` (the repository folder on Arrhenius). Tables go to `results/cloud_heights/`, figures to `figures/cloud_heights/` (or next to the tables if there is no `figures` folder). It only reads the built files, so it is quick.

In [ ]:
import sys, glob, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", message="All-NaN slice")
warnings.filterwarnings("ignore", message="Mean of empty slice")

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
import arm_ena_nd_pnsd as m

OUT = ROOT / "results" / "cloud_heights"
OUT.mkdir(parents=True, exist_ok=True)
FIG = ROOT / "figures" / "cloud_heights" if (ROOT / "figures").is_dir() else OUT
FIG.mkdir(parents=True, exist_ok=True)

MIN_CF = 0.5                       # in-cloud hours, as in the quicklooks
MAX_BASE, MAX_TOP = 2000.0, 3000.0 # the standard height screen
COUPLED_DZ = 150.0                 # m, cloud base - LCL below this counts as coupled
N_BOOT = 100                       # day-block resamples for the tercile fits (0 = no intervals)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
C_BASE, C_TOP = "#3B6FB6", "#D9822B"


## Sites

Ground sites are listed explicitly; aircraft files are picked up from `results/aircraft/`. Missing files are skipped, so this can be rerun as new sites are built.

In [ ]:
R = lambda *p: ROOT.joinpath("results", *p)
GROUND = [  # key, label, built file, height source
    ("ena_uhsas", "ENA (NDROP)", R("ena", "ena_2014_2023_uhsas.nc"), "NDROP"),
    ("sgp", "SGP (NDROP)", R("sgp", "sgp_2016_2025_smps.nc"), "NDROP"),
    ("lasic", "LASIC (NDROP)", R("lasic", "lasic_2016_2017_uhsas.nc"), "NDROP"),
    ("epc_pier", "EPCAPE pier (NDROP)", R("epcape_pier", "epc_2023_2024_smps.nc"), "NDROP"),
    ("sgp_cloudnet", "SGP (Cloudnet)", R("sgp", "sgp_cloudnet.nc"), "Cloudnet"),
    ("hyytiala", "Hyytiälä", R("hyytiala", "hyytiala_cloudnet.nc"), "Cloudnet"),
    ("kenttarova", "Kenttärova", R("kenttarova", "kenttarova_cloudnet.nc"), "Cloudnet"),
    ("ny-alesund", "Ny-Ålesund", R("ny-alesund", "ny-alesund_cloudnet.nc"), "Cloudnet"),
    ("mace-head", "Mace Head", R("mace-head", "mace-head_cloudnet.nc"), "Cloudnet"),
    ("cabauw", "Cabauw", R("cabauw", "cabauw_cloudnet.nc"), "Cloudnet"),
    ("granada", "Granada", R("granada", "granada_cloudnet.nc"), "Cloudnet"),
    ("palaiseau", "Palaiseau", R("palaiseau", "palaiseau_cloudnet.nc"), "Cloudnet"),
    ("schneefernerhaus", "Schneefernerhaus", R("schneefernerhaus", "schneefernerhaus_cloudnet.nc"), "Cloudnet"),
    ("oliktok", "Oliktok", R("oliktok", "oliktok_cloudnet.nc"), "Cloudnet"),
    ("bankhead", "Bankhead", R("bankhead", "bankhead_cloudnet.nc"), "Cloudnet"),
]
AIRCRAFT = sorted(glob.glob(str(R("aircraft", "*_aircraft*.nc"))))

def load_ground(path):
    with xr.open_dataset(path) as ds:
        if "cloud_base_height" not in ds:
            return None
        i100 = int(np.argmin(np.abs(ds["diameter"].values - 100)))
        get = lambda v: ds[v].values.astype(float) if v in ds else np.full(ds.sizes["time"], np.nan)
        d = pd.DataFrame({"base": get("cloud_base_height"), "top": get("cloud_top_height"),
                          "thick": get("cloud_thickness"), "dz": get("decoupling_dz"),
                          "cf": get("cloud_frac"), "Nd": get("CDNC"), "N100": ds["CCN"].values[:, i100]},
                         index=pd.DatetimeIndex(ds["time"].values))
        ccn, diam = ds["CCN"].values, ds["diameter"].values
    if d["thick"].isna().all():
        d["thick"] = d["top"] - d["base"]
    return d, ccn, diam

SITES = {}
for key, label, path, src in GROUND:
    if not Path(path).exists():
        print(f"skip {label:22s} (no built file yet)")
        continue
    r = load_ground(path)
    if r is None:
        print(f"skip {label:22s} (no cloud heights in file)")
        continue
    d, ccn, diam = r
    keep = d["cf"].to_numpy() >= MIN_CF
    SITES[key] = dict(label=label, src=src, d=d[keep], ccn=ccn[keep], diam=diam)
print(f"\n{len(SITES)} ground sites with cloud heights; {len(AIRCRAFT)} aircraft files")

## 1 · Summary table

In [ ]:
def q(x, f="{:.0f}"):
    x = pd.Series(x).dropna()
    return f"{f.format(x.median())} [{f.format(x.quantile(.25))}–{f.format(x.quantile(.75))}]" if len(x) else "–"

rows = []
for k, s in SITES.items():
    d = s["d"]
    screened = (d["base"] <= MAX_BASE) & (d["top"] <= MAX_TOP)
    row = {"site": s["label"], "heights from": s["src"], "hours": len(d),
           "base, m": q(d["base"]), "top, m": q(d["top"]), "thickness, m": q(d["thick"]),
           "% removed by height screen": round(100 * (1 - screened.mean()), 1),
           "base, screened hours": q(d.loc[screened, "base"])}
    if d["dz"].notna().any():
        row["base − LCL, m"] = q(d["dz"])
        row["% coupled"] = round(100 * (d["dz"] <= COUPLED_DZ).sum() / d["dz"].notna().sum(), 1)
    rows.append(row)
summary = pd.DataFrame(rows).set_index("site")
summary.to_csv(OUT / "cloud_heights_summary.csv")
summary

## 2 · Base and top at each site

Boxes span the 25th–75th percentile, whiskers 5th–95th, for in-cloud hours (cloud fraction ≥ 0.5) before the height screen. Dashed lines mark the screen limits.

In [ ]:
keys = list(SITES)
fig, ax = plt.subplots(figsize=(9, 0.45 * len(keys) + 1.5))
pos = np.arange(len(keys))
for off, var, col, lab in ((-0.18, "base", C_BASE, "cloud base"), (0.18, "top", C_TOP, "cloud top")):
    data = [SITES[k]["d"][var].dropna().to_numpy() for k in keys]
    kw = dict(positions=pos + off, widths=0.3, whis=(5, 95), showfliers=False,
              patch_artist=True, medianprops=dict(color="k"))
    try:
        bp = ax.boxplot(data, orientation="horizontal", **kw)   # matplotlib >= 3.10
    except TypeError:
        bp = ax.boxplot(data, vert=False, **kw)                 # older matplotlib
    for b in bp["boxes"]:
        b.set(facecolor=col, alpha=0.6, edgecolor=col)
    ax.plot([], [], "s", color=col, alpha=0.6, label=lab)
ax.axvline(MAX_BASE, color=C_BASE, ls="--", lw=1)
ax.axvline(MAX_TOP, color=C_TOP, ls="--", lw=1)
ax.set_yticks(pos)
ax.set_yticklabels([f"{SITES[k]['label']} ({len(SITES[k]['d'])} h)" for k in keys])
ax.invert_yaxis()
ax.set_xlabel("Height above ground, m")
ax.set_xlim(0, 6000)
ax.legend(frameon=False, loc="lower right")
ax.set_title("Cloud base and top, in-cloud hours", loc="left")
fig.tight_layout()
fig.savefig(FIG / "cloud_base_top_by_site.png", dpi=150)
plt.show()
plt.close(fig)

Histograms of cloud base (blue) and top (orange) per site, 0–6 km in 100 m bins. Shading marks the heights the standard screen removes.

In [ ]:
n = len(keys); ncol = 3; nrow = int(np.ceil(n / ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(4.4 * ncol, 2.8 * nrow), squeeze=False, sharex=True)
bins = np.arange(0, 6100, 100)
for ax, k in zip(axs.flat, keys):
    d = SITES[k]["d"]
    ax.hist(d["base"].dropna(), bins=bins, color=C_BASE, alpha=0.7, label="base")
    ax.hist(d["top"].dropna(), bins=bins, color=C_TOP, alpha=0.5, label="top")
    ax.axvspan(MAX_BASE, 6000, color="0.85", zorder=0)
    ax.set_title(SITES[k]["label"], loc="left", fontsize=10)
    ax.set_ylabel("hours")
for ax in axs[-1]:
    ax.set_xlabel("Height above ground, m")
for ax in list(axs.flat)[n:]:
    ax.set_visible(False)
axs.flat[0].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "cloud_height_histograms.png", dpi=150)
plt.show()
plt.close(fig)

## 3 · Seasonal cycle of cloud base

Monthly median cloud base with IQR, all years pooled. Months with fewer than 20 in-cloud hours are left out.

In [ ]:
fig, axs = plt.subplots(nrow, ncol, figsize=(4.4 * ncol, 2.6 * nrow), squeeze=False, sharex=True)
months = np.arange(1, 13)
for ax, k in zip(axs.flat, keys):
    b = SITES[k]["d"]["base"].dropna()
    g = b.groupby(b.index.month)
    st = pd.DataFrame({"med": g.median(), "q25": g.quantile(.25), "q75": g.quantile(.75), "n": g.size()}).reindex(months)
    st.loc[st["n"] < 20, ["med", "q25", "q75"]] = np.nan
    ax.fill_between(months, st["q25"], st["q75"], color=C_BASE, alpha=0.25, lw=0)
    ax.plot(months, st["med"], "o-", color=C_BASE, ms=3)
    ax.axhline(MAX_BASE, color="0.5", ls="--", lw=0.8)
    ax.set_title(SITES[k]["label"], loc="left", fontsize=10)
    ax.set_ylabel("cloud base, m")
    ax.set_xticks(months, list("JFMAMJJASOND"))
for ax in list(axs.flat)[n:]:
    ax.set_visible(False)
fig.tight_layout()
fig.savefig(FIG / "cloud_base_seasonal_cycle.png", dpi=150)
plt.show()
plt.close(fig)

## 4 · Coupling to the surface (ARM sites)

Cloud base minus the surface lifting condensation level (LCL). Near zero means the cloud sits on a well-mixed boundary layer, so surface aerosol is a fair proxy for what the cloud ingests; large positive values mean a decoupled cloud above the surface layer. The dashed line is the 150 m threshold used for the coupled/decoupled split.

In [ ]:
dz_keys = [k for k in keys if SITES[k]["d"]["dz"].notna().sum() > 20]
if dz_keys:
    fig, ax = plt.subplots(figsize=(8, 3.2))
    bins = np.arange(-1000, 3050, 50)
    for k, col in zip(dz_keys, plt.cm.tab10.colors):
        dz = SITES[k]["d"]["dz"].dropna()
        ax.hist(dz, bins=bins, histtype="step", lw=1.6, color=col, density=True,
                label=f"{SITES[k]['label']}: {100 * (dz <= COUPLED_DZ).mean():.0f}% coupled")
    ax.axvline(COUPLED_DZ, color="k", ls="--", lw=1)
    ax.set_xlabel("cloud base − surface LCL, m"); ax.set_ylabel("density")
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG / "decoupling_by_site.png", dpi=150)
    plt.show()
    plt.close(fig)
else:
    print("No site with decoupling_dz (build the ARM sites with --met-dir).")

## 5 · Susceptibility by cloud-base height

At each site, in-cloud hours that pass the height screen are split into terciles of cloud base, and S(100 nm) is refitted in each with the same binned-median method. Intervals are day-block bootstrap (set `N_BOOT = 0` to skip them). Terciles with fewer than 60 hours are left out.

In [ ]:
i_rows = []
for k in keys:
    s = SITES[k]; d = s["d"]
    ok = ((d["base"] <= MAX_BASE) & (d["top"] <= MAX_TOP)).to_numpy() & np.isfinite(d["base"].to_numpy())
    if ok.sum() < 180:
        continue
    edges = np.nanpercentile(d["base"].to_numpy()[ok], [0, 100 / 3, 200 / 3, 100])
    i100 = int(np.argmin(np.abs(s["diam"] - 100)))
    for t in range(3):
        sel = ok & (d["base"].to_numpy() >= edges[t]) & (d["base"].to_numpy() <= edges[t + 1])
        if sel.sum() < 60:
            continue
        y, c, tt = d["Nd"].to_numpy()[sel], s["ccn"][sel], d.index[sel]
        tab, dd = m.susceptibility_table(c, y, s["diam"])
        lo = hi = np.nan
        if N_BOOT:
            sb, _ = m.bootstrap_by_day(tt, c, y, s["diam"], N_BOOT, cut_idx=[i100])
            lo, hi = m._ci(sb[:, i100])
        i_rows.append({"site": s["label"], "tercile": ["low", "mid", "high"][t],
                       "base range, m": f"{edges[t]:.0f}–{edges[t + 1]:.0f}",
                       "median base, m": float(np.median(d["base"].to_numpy()[sel])), "hours": int(sel.sum()),
                       "S(100 nm)": tab["S"].iloc[i100], "S lo": lo, "S hi": hi, "closure, nm": dd})
terc = pd.DataFrame(i_rows)
terc.to_csv(OUT / "susceptibility_by_cloud_base_tercile.csv", index=False)

if len(terc):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    marks = "osD^vP*Xph"
    for j, (site, g) in enumerate(terc.groupby("site", sort=False)):
        col = plt.cm.tab10.colors[j % 10]
        ax.errorbar(g["median base, m"], g["S(100 nm)"],
                    yerr=[g["S(100 nm)"] - g["S lo"], g["S hi"] - g["S(100 nm)"]] if N_BOOT else None,
                    fmt=marks[j // 10 % len(marks)] + "-", color=col, ms=5, capsize=2, lw=1.2, label=site)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xlabel("median cloud base of the tercile, m above ground")
    ax.set_ylabel("S(100 nm)")
    ax.legend(frameon=False, fontsize=7, loc="center left", bbox_to_anchor=(1.01, 0.5))
    ax.set_title("Susceptibility in cloud-base terciles", loc="left")
    fig.tight_layout()
    fig.savefig(FIG / "susceptibility_by_cloud_base.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
terc.round(2)

## 6 · Aircraft campaigns

For each campaign built with `aircraft_build.py`: the local cloud base of the near-base passes (metres above sea level) and how far above base each pass was. Passes are limited to 300 m above base by the builder.

In [ ]:
if AIRCRAFT:
    a_rows = []
    fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
    for f, col in zip(AIRCRAFT, plt.cm.tab10.colors):
        with xr.open_dataset(f) as ds:
            name = Path(f).stem.replace("_aircraft", "")
            base, above = ds["cloud_base"].values, ds["above_base"].values
            a_rows.append({"campaign": name, "passes": len(base), "flights": pd.DatetimeIndex(ds["time"].values).normalize().nunique(),
                           "cloud base MSL, m": q(base), "pass height above base, m": q(above),
                           "aerosol sampled below base, m": q(ds["cloud_base"].values - ds["aer_alt"].values)})
        axs[0].hist(base, bins=np.arange(0, 4100, 100), histtype="step", lw=1.6, color=col, label=name)
        axs[1].hist(above, bins=np.arange(-50, 325, 25), histtype="step", lw=1.6, color=col, label=name)
    axs[0].set_xlabel("local cloud base, m above sea level"); axs[0].set_ylabel("passes")
    axs[1].set_xlabel("pass altitude above local cloud base, m")
    axs[0].legend(frameon=False, fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG / "aircraft_cloud_base.png", dpi=150)
    plt.show()
    plt.close(fig)
    air = pd.DataFrame(a_rows).set_index("campaign")
    air.to_csv(OUT / "aircraft_cloud_heights.csv")
    display(air)
else:
    print("No aircraft files in results/aircraft yet.")

## Notes

- **NDROP heights** are the cloud base and top used in the retrieval (from ARSCL), averaged over the valid retrievals in each hour; NDROP only retrieves daytime overcast liquid cloud, so very high or multi-layer cloud is already rare.
- **Cloudnet heights** come from pixels with a reliable droplet retrieval only. Cloud top is therefore the top of the retrievable liquid and thin layers can look thinner than they are; the Cloudnet build already limits base to 2.5 km.
- **Height screen:** base ≤ 2 km and top ≤ 3 km is what the quicklooks use to keep boundary-layer cloud. At SGP it removes most summer cloud, which is mid-level.
- **Terciles** are site-specific, so "low" at SGP and "low" at LASIC are different heights; compare the trend within a site, and the median base on the x-axis across sites.
- **Aircraft** heights are above sea level; ENA, ACAPEX and ACME-V flights are over sea, HI-SCALE over ground at about 315 m, CACTI over terrain of 1–2 km.